Customer Churn Prediction • • • • • • Problem Statement: Predict whether a customer is likely to stop using a telecom, banking or subscription service. Suggested Dataset: Use the IBM Telco Customer Churn dataset or a similar public dataset. Student Tasks: Study churn distribution Clean and encode customer features Train Logistic Regression and Random Forest models Evaluate accuracy, precision, recall and F1-score Create a confusion matrix Identify important churn factors Expected Learning Outcomes: Classification, Imbalanced data awareness, Evaluation metrics, Feature importance, Business interpretation.

In [ ]:
"""
Generates a realistic synthetic Telco Customer Churn dataset.
Schema matches the popular IBM/Kaggle Telco Customer Churn dataset,
so this script can be swapped out later with the real CSV
(just replace the file path in 01_churn_prediction.py).
"""

import numpy as np
import pandas as pd

np.random.seed(42)
N = 7043  # same size as the real IBM Telco dataset

# --- Demographics ---
gender = np.random.choice(['Male', 'Female'], N)
senior_citizen = np.random.choice([0, 1], N, p=[0.84, 0.16])
partner = np.random.choice(['Yes', 'No'], N, p=[0.48, 0.52])
dependents = np.random.choice(['Yes', 'No'], N, p=[0.30, 0.70])

# --- Account info ---
tenure = np.random.randint(0, 73, N)
contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], N, p=[0.55, 0.21, 0.24])
paperless_billing = np.random.choice(['Yes', 'No'], N, p=[0.59, 0.41])
payment_method = np.random.choice(
    ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
    N, p=[0.34, 0.23, 0.22, 0.21]
)

# --- Services ---
phone_service = np.random.choice(['Yes', 'No'], N, p=[0.90, 0.10])
multiple_lines = np.where(
    phone_service == 'No', 'No phone service',
    np.random.choice(['Yes', 'No'], N, p=[0.42, 0.58])
)
internet_service = np.random.choice(['DSL', 'Fiber optic', 'No'], N, p=[0.34, 0.44, 0.22])

def dependent_service(internet, p_yes=0.5):
    return np.where(
        internet == 'No', 'No internet service',
        np.random.choice(['Yes', 'No'], N, p=[p_yes, 1 - p_yes])
    )

online_security = dependent_service(internet_service, 0.35)
online_backup = dependent_service(internet_service, 0.40)
device_protection = dependent_service(internet_service, 0.40)
tech_support = dependent_service(internet_service, 0.35)
streaming_tv = dependent_service(internet_service, 0.45)
streaming_movies = dependent_service(internet_service, 0.45)

# --- Charges ---
base_charge = 18.0
internet_charge = np.select(
    [internet_service == 'DSL', internet_service == 'Fiber optic', internet_service == 'No'],
    [25, 45, 0]
)
extra_services = sum(
    np.where(s == 'Yes', np.random.uniform(4, 9, N), 0)
    for s in [online_security, online_backup, device_protection, tech_support, streaming_tv, streaming_movies]
)
phone_charge = np.where(phone_service == 'Yes', np.random.uniform(15, 25, N), 0)

monthly_charges = np.round(base_charge + internet_charge + extra_services + phone_charge, 2)
total_charges = np.round(monthly_charges * tenure + np.random.normal(0, 20, N), 2)
total_charges = np.clip(total_charges, 0, None)

# --- Churn logic (realistic: driven by contract, tenure, charges, internet type) ---
churn_score = (
    (contract == 'Month-to-month') * 2.0
    + (contract == 'One year') * 0.4
    + (internet_service == 'Fiber optic') * 1.0
    + (payment_method == 'Electronic check') * 0.8
    + (tenure < 12) * 1.5
    + (tenure > 48) * -1.5
    + (online_security == 'No') * 0.4
    + (tech_support == 'No') * 0.4
    + (monthly_charges > 80) * 0.6
    + (senior_citizen == 1) * 0.3
    + (partner == 'No') * 0.3
    + np.random.normal(0, 1.2, N)
)
churn_prob = 1 / (1 + np.exp(-(churn_score - 4.2)))
churn = np.where(np.random.uniform(0, 1, N) < churn_prob, 'Yes', 'No')

df = pd.DataFrame({
    'customerID': [f'{7000+i:04d}-{"".join(np.random.choice(list("ABCDEFGH"), 5))}' for i in range(N)],
    'gender': gender,
    'SeniorCitizen': senior_citizen,
    'Partner': partner,
    'Dependents': dependents,
    'tenure': tenure,
    'PhoneService': phone_service,
    'MultipleLines': multiple_lines,
    'InternetService': internet_service,
    'OnlineSecurity': online_security,
    'OnlineBackup': online_backup,
    'DeviceProtection': device_protection,
    'TechSupport': tech_support,
    'StreamingTV': streaming_tv,
    'StreamingMovies': streaming_movies,
    'Contract': contract,
    'PaperlessBilling': paperless_billing,
    'PaymentMethod': payment_method,
    'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges,
    'Churn': churn
})

df.to_csv('/content/Telco_customer_churn.csv', index=False)
print("Dataset created:", df.shape)
print(df['Churn'].value_counts(normalize=True))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    precision_recall_curve
)

# ---- Style ----
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'
COLORS = ['skyblue', 'salmon']   # blue = stay, red = churn
OUT = '/content/'

# ====================================================================
# 1. LOAD DATA
# ====================================================================
df = pd.read_csv(OUT + 'Telco_customer_churn.csv')
print("Shape:", df.shape)
print(df.head())

# ====================================================================
# 2. CLEANING
# ====================================================================
# TotalCharges sometimes has blanks/whitespace in the real dataset -> coerce to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Drop ID column (not predictive)
df.drop(columns=['customerID'], inplace=True)

print("\nMissing values:\n", df.isnull().sum().sum())

# ==================================================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ==================================================================

# --- 3.1 Churn distribution ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=COLORS, startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Churn Distribution', fontsize=13, fontweight='bold')

sns.countplot(data=df, x='Churn', hue='Churn', palette=COLORS, legend=False, ax=axes[1])
axes[1].set_title('Churn Count', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
for p in axes[1].patches:
    axes[1].annotate(str(int(p.get_height())),
                      (p.get_x() + p.get_width()/2, p.get_height()),
                      ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show() # Display plot directly

# --- 3.2 Churn by Contract type & Tenure ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', stacked=True, color=COLORS, ax=axes[0])
axes[0].set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('% of customers')
axes[0].legend(title='Churn')
axes[0].tick_params(axis='x', rotation=20)

sns.histplot(data=df, x='tenure', hue='Churn', bins=30, palette=COLORS,
             multiple='stack', ax=axes[1])
axes[1].set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show() # Display plot directly

# --- 3.3 Monthly charges vs churn ---
plt.figure(figsize=(7, 4.5))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', hue='Churn', palette=COLORS, legend=False)
plt.title('Monthly Charges vs Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show() # Display plot directly

print("EDA plots displayed.")



In [ ]:
# ====================================================================
# 4. ENCODING
# ====================================================================
df_model = df.copy()

binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0})

df_model['gender'] = df_model['gender'].map({'Male': 1, 'Female': 0})
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

# Multi-category columns -> one-hot encoding
multi_cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
                   'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
                   'Contract', 'PaymentMethod']

df_model = pd.get_dummies(df_model, columns=multi_cat_cols, drop_first=True)

print("\nEncoded shape:", df_model.shape)

In [ ]:
# ====================================================================
# 5. TRAIN / TEST SPLIT + SCALING
# ====================================================================
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nTrain size: ", X_train.shape[0], "  |  Test size: ", X_test.shape[0])
print("Train churn rate: ", '{:.2%}'.format(y_train.mean()), "  |  Test churn rate: ", '{:.2%}'.format(y_test.mean()))

In [ ]:
# ====================================================================
# 6. MODEL TRAINING
# ====================================================================
# Logistic Regression (needs scaled features) - class_weight handles imbalance
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Random Forest (doesn't need scaling) - class_weight handles imbalance
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                             random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

models = {
    'Logistic Regression': (log_reg, X_test_scaled),
    'Random Forest': (rf, X_test)
}


In [ ]:
# ====================================================================
# 7. EVALUATION
# ====================================================================
results = []
preds_store = {}

for name, (model, X_te) in models.items():
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    preds_store[name] = (y_pred, y_prob)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index('Model').round(3)
print("\n===== MODEL COMPARISON ===telek")
print(results_df)


# --- 7.1 Metrics bar chart ---
plt.figure(figsize=(9, 5))
results_df.plot(kind='bar', ax=plt.gca(), color=sns.color_palette('viridis', 5))
plt.title('Model Performance Comparison', fontsize=13, fontweight='bold')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1.05)
plt.legend(loc='lower right', ncol=2)
plt.tight_layout()
plt.show() # Display plot directly

# --- 7.2 Confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, (y_pred, _)) in zip(axes, preds_store.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show() # Display plot directly

# --- 7.3 ROC curves ---
plt.figure(figsize=(6.5, 5.5))
for name, (_, y_prob) in preds_store.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show() # Display plot directly

print("\nDetailed classification reports:\n")
for name, (y_pred, _) in preds_store.items():
    print(f"--- {name} ---")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))


In [ ]:
# ====================================================================
# 8. FEATURE IMPORTANCE (business interpretation)
# ====================================================================
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(12)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index,
            palette='rocket', legend=False)
plt.title('Top 12 Churn Drivers (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show() # Display plot directly

print("\nTop churn factors:\n", importances)

# Logistic Regression coefficients (direction of effect)
coef = pd.Series(log_reg.coef_[0], index=X.columns).sort_values()
top_coef = pd.concat([coef.head(6), coef.tail(6)])

plt.figure(figsize=(8, 6))
colors_coef = ['skyblue' if v < 0 else 'salmon' for v in top_coef.values]
plt.barh(top_coef.index, top_coef.values, color=colors_coef)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression: Factors Increasing/Decreasing Churn',
          fontsize=12, fontweight='bold')
plt.xlabel('Coefficient (red = increases churn, blue = decreases churn)')
plt.tight_layout()
plt.show() # Display plot directly


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# Reset index to make 'Model' a regular column for Plotly Express
results_df_plot = results_df.reset_index()

fig = px.bar(
    results_df_plot,
    x='Model',
    y=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    barmode='group',
    title='Model Performance Comparison (Plotly)',
    labels={'value': 'Score', 'variable': 'Metric'},
    color_discrete_sequence=px.colors.qualitative.Plotly # Use a Plotly color sequence
)

fig.update_layout(yaxis_range=[0, 1.05]) # Set y-axis range
fig.show()